## 1. Parâmetros e Configuração do Ambiente

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from datetime import datetime
from pyspark.sql import Row

catalog = "cinedata_analytics"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

DataFrame[]

## 2. Funções de Data Quality

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas")

## 3. Funções _Utilitárias_

Funções reutilizáveis aplicadas em todas as transformações Silver:

* **filter_valid_keys**: Remove linhas com chave NULL, vazia ou apenas whitespace.
* **sanitize_text**: Remove caracteres não-alfabéticos, achata espaços e converte para minúsculas.
* **deduplicate_by_ingestion**: Mantém apenas o registro mais recente por chave (window + row_number).
* **treat_textual_nulls**: Converte sentinelas textuais ("N/A", "unknown", etc.) para NULL de forma case-insensitive.
* **parse_financial_value**: Limpa strings monetárias (remove símbolos, interpreta sufixos M/K) e converte para decimal.
* **explode_entities**: Extrai, limpa, explode e padroniza colunas de entidades (cast, directors, writers, production_companies).

In [0]:
def filter_valid_keys(df, key_col: str):
    """Remove linhas cujo valor da coluna seja NULL, Whitespace ou String Vazia."""
    return df.filter(F.col(key_col).isNotNull() & (F.trim(F.col(key_col)) != ""))

def sanitize_text(col_name: str):
    """Remove caracteres não-alfabéticos, achata espaços entre palavras, converte em minúsculas e aplica trim."""
    return F.trim(
        F.lower(
            F.regexp_replace(
                F.regexp_replace(col_name, r"[^a-zA-Z\s]", " "), 
                r"\s+", " "
            )
        )
    )

def deduplicate_by_ingestion(df, partition_cols: list, order_col: str = "ingestion_datetime"):
    """
    Deduplica um DataFrame mantendo estritamente o registro mais recente.
    """
    window_dedup = Window.partitionBy(*partition_cols).orderBy(F.col(order_col).desc())
    
    return (
        df.withColumn("row_num", F.row_number().over(window_dedup))
        .filter(F.col("row_num") == 1)
        .drop("row_num", order_col)
    )

def treat_textual_nulls(col_name: str, null_terms: list):
    """
    Converte valores textuais conhecidos (case-insensitive) de ausência de dados para NULL (SQL).
    """
    lower_null_terms = [term.lower() for term in null_terms]
    
    return F.when(
        F.lower(F.trim(F.col(col_name))).isin(lower_null_terms), 
        F.lit(None).cast("string")
    ).otherwise(F.col(col_name))

def parse_financial_value(col_name: str):
    """
    Higieniza e converte strings monetárias com extrema resiliência.
    1. Trator: Remove QUALQUER caractere que não seja dígito, ponto, M, K ou sinal negativo (destrói €, £, $, letras aleatórias).
    2. Bisturi: Avalia sufixos únicos (M/K). Anomalias estruturais viram NULL.
    3. Converte para Decimal(18,2) e anula valores <= 0.
    """
    texto_limpo = F.regexp_replace(F.upper(F.col(col_name)), r"[^\d\.MK\-]", "")
    
    numero_str = F.regexp_extract(texto_limpo, r"^(\-?\d+\.?\d*)([MK]?)$", 1)
    multiplicador = F.regexp_extract(texto_limpo, r"^(\-?\d+\.?\d*)([MK]?)$", 2)
    
    numero_base = numero_str.cast("decimal(28,6)")
    
    valor_final = (
        F.when(multiplicador == "M", numero_base * 1000000)
         .when(multiplicador == "K", numero_base * 1000)
         .otherwise(numero_base)
    )
    
    return F.when(valor_final > 0, valor_final.cast("decimal(18,2)")) \
            .otherwise(F.lit(None).cast("decimal(18,2)"))

def explode_entities(df, col_name, tipo_entidade):
    """Extrai, limpa e explode uma coluna de entidades. Remove sentinelas (N/A, [], Nenhum),
    resíduos de column shift (números, paths de imagem) e padroniza capitalização via initcap."""
    return (
        df.select(F.col("id").alias("id_filme"), F.col(col_name).alias("entidade_raw"))
        .withColumn("entidade_clean",
            F.when(F.col("entidade_raw").isin("N/A", "[]", "Nenhum", "", "null"), F.lit(None))
             .otherwise(F.col("entidade_raw"))
        )
        .filter(F.col("entidade_clean").isNotNull())
        .withColumn("entidade", F.explode(F.split("entidade_clean", ",")))
        .withColumn("nome_entidade", F.trim(F.col("entidade")))
        .filter(F.col("nome_entidade") != "")
        .filter(~F.col("nome_entidade").rlike(r"^\d+$"))
        .filter(~F.col("nome_entidade").rlike(r"(?i)^/.*\.(jpe?g|png|gif|bmp|webp|svg|tiff?)$"))
        .withColumn("nome_entidade", F.initcap(F.col("nome_entidade")))
        .withColumn("tipo_entidade", F.lit(tipo_entidade))
        .select("id_filme", "nome_entidade", "tipo_entidade")
    )
    

## 4. Tabela cotação

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

print("\nProcessando: tb_cotacao_dolar...")

df_dolar_raw = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

df_dolar_clean_temp = (
    df_dolar_raw
    .withColumn("data_cotacao", F.to_date("dataHoraCotacao"))
    .withColumn("cotacao_compra", F.col("cotacaoCompra").cast("decimal(10,4)"))
)

df_dolar_dedup = deduplicate_by_ingestion(df_dolar_clean_temp, partition_cols=["data_cotacao"])
df_dolar_clean = df_dolar_dedup.select("data_cotacao", "cotacao_compra")

limites = df_dolar_clean.agg(F.min("data_cotacao").alias("inicio")).first()
data_inicio = limites["inicio"]

df_calendario = spark.sql(f"""
    SELECT explode(sequence(DATE'{data_inicio}', CURRENT_DATE(), INTERVAL 1 DAY)) AS data_referencia
""")

df_dolar_continuo = (
    df_calendario
    .join(df_dolar_clean, F.col("data_referencia") == F.col("data_cotacao"), "left")
    .drop("data_cotacao")
)

window_ordem_ffill = Window.partitionBy(F.lit(1)).orderBy("data_referencia")

df_grupos = (
    df_dolar_continuo
    .withColumn("grp_ffill", F.count("cotacao_compra").over(window_ordem_ffill))
)

window_ffill = Window.partitionBy("grp_ffill")

df_dolar_silver = (
    df_grupos
    .withColumn("ffill", F.max("cotacao_compra").over(window_ffill))
    .drop("cotacao_compra", "grp_ffill")
    .withColumnRenamed("ffill", "cotacao_compra")
)

dq_check_unique("tb_cotacao_dolar", "Unicidade de data_referencia", df_dolar_silver, ["data_referencia"])
dq_check("tb_cotacao_dolar", "Cotação válida (nula ou > 0)", df_dolar_silver, F.col("cotacao_compra").isNull() | (F.col("cotacao_compra") > 0))

tabela_destino_dolar = f"{silver_schema}.tb_cotacao_dolar"

(
    df_dolar_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_destino_dolar)
)

print(f"✅ Tabela gravada com sucesso em: {tabela_destino_dolar}")
display(df_dolar_silver.orderBy("data_referencia"))


Processando: tb_cotacao_dolar...
[✅ PASS] tb_cotacao_dolar | Unicidade de data_referencia | 0 chaves duplicadas
[✅ PASS] tb_cotacao_dolar | Cotação válida (nula ou > 0) | 0/48 linhas falharam
✅ Tabela gravada com sucesso em: cinedata_analytics.silver.tb_cotacao_dolar


data_referencia,cotacao_compra
2026-08-05,5.1148
2026-08-06,5.1011
2026-08-07,5.0902
2026-08-08,5.0902
2026-08-09,5.0902
2026-08-10,5.0957
2026-08-11,5.1279
2026-08-12,5.1632
2026-08-13,5.1853
2026-08-14,5.2230


## 5. Tratamento: tb_movies_info -> tb_info_filmes

- **Deduplicação**: Remove registros com `id` duplicados, mantendo o registro com a `ingestion_datetime` mais recente.
- **Limpeza e Tradução do Status**: Executa a limpeza da coluna `status` utilizando a Função Utilitária `sanitize_text` e aplica mapeamento dos valores para português (Released → Lançado; Post Production
→ Pós-Produção; In Production → Em Produção; Planned → Planejado; Rumored → Rumores; Canceled
→ Cancelado).
- **Tratamento Multi-Formato de Datas**: Conversão sequencial de strings para datas, com suporte aos padrões `yyyy-MM-dd`, `dd/MM/yyyy` e `MM-dd-yyyy`. Valores incompatíveis com todos os formatos validados são padronizados como `NULL`.
- **Coluna Derivada**: Enriquece a tabela `silver.tb_info_filmes` com a coluna `ano_lancamento`, extraída de `data_lancamento`.


In [0]:
print("\nProcessando: tb_info_filmes...")

df_info_raw = spark.table(f"{bronze_schema}.tb_movies_info")
df_info_raw = filter_valid_keys(df_info_raw, "id")

df_info_dedup = deduplicate_by_ingestion(df_info_raw, partition_cols=["id"])

df_info_silver = (
    df_info_dedup
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
    
    .withColumn("data_lancamento", F.coalesce(
        F.expr("TRY_TO_DATE(release_date, 'yyyy-MM-dd')"),
        F.expr("TRY_TO_DATE(release_date, 'dd/MM/yyyy')"),
        F.expr("TRY_TO_DATE(release_date, 'MM-dd-yyyy')")
    ))
    .withColumn("ano_lancamento", F.year("data_lancamento"))
    
    .withColumn("duracao_minutos", F.expr("TRY_CAST(duracao_minutos AS INT)"))
    
    .withColumn("status_norm", sanitize_text("status"))
    .withColumn("status_filme",
        F.when(F.col("status_norm") == "released", "Lançado")
         .when(F.col("status_norm") == "post production", "Pós-Produção")
         .when(F.col("status_norm") == "in production", "Em Produção")
         .when(F.col("status_norm") == "planned", "Planejado")
         .when(F.col("status_norm") == "rumored", "Rumores")
         .when(F.col("status_norm") == "canceled", "Cancelado")
         .otherwise("Não Informado")
    )
    .drop("release_date", "status", "status_norm")
)

dq_check_unique("tb_info_filmes", "Unicidade de id_filme", df_info_silver, ["id_filme"])
dq_check("tb_info_filmes", "Duração válida (nula ou >=0)", df_info_silver, F.col("duracao_minutos").isNull() | (F.col("duracao_minutos") >= 0))
dq_check("tb_info_filmes", "Ano de lançamento válido (nulo ou >=1900)", df_info_silver, F.col("ano_lancamento").isNull() | (F.col("ano_lancamento") >= 1900))

tabela_destino_info = f"{silver_schema}.tb_info_filmes"

(
    df_info_silver.write
    .format("delta")
    .mode("overwrite") 
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_destino_info)
)

print(f"✅ Tabela gravada com sucesso em: {tabela_destino_info}")


Processando: tb_info_filmes...
[✅ PASS] tb_info_filmes | Unicidade de id_filme | 0 chaves duplicadas
[✅ PASS] tb_info_filmes | Duração válida (nula ou >=0) | 0/93220 linhas falharam
[✅ PASS] tb_info_filmes | Ano de lançamento válido (nulo ou >=1900) | 0/93220 linhas falharam
✅ Tabela gravada com sucesso em: cinedata_analytics.silver.tb_info_filmes


## 6. Tratamento: tb_movies_financials -> tb_financeiro_filmes

In [0]:
print("\nProcessando: tb_financeiro_filmes...")

# 1. Leitura com Barreira de Nulos Estruturais Aplicada
df_fin_raw = spark.table(f"{bronze_schema}.tb_movies_financials")
df_fin_raw = filter_valid_keys(df_fin_raw, "id")

termos_nulos_financeiros = ["unknown", "não informado", "nao informado", "n/a", ""]

df_fin_dedup = deduplicate_by_ingestion(df_fin_raw, partition_cols=["id"])

df_fin_silver_parcial = (
    df_fin_dedup
    .withColumnRenamed("id", "id_filme")
    
    .withColumn("budget_null_tratado", treat_textual_nulls("budget", termos_nulos_financeiros))
    .withColumn("revenue_null_tratado", treat_textual_nulls("revenue", termos_nulos_financeiros))
    
    .withColumn("orcamento_usd", parse_financial_value("budget_null_tratado"))
    .withColumn("receita_usd", parse_financial_value("revenue_null_tratado"))
    
    .drop("budget", "revenue", "budget_null_tratado", "revenue_null_tratado")
)

df_dolar_raw = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

df_dolar = df_dolar_raw.select(
    F.to_date("dataHoraCotacao").alias("data_cotacao"),
    F.col("cotacaoCompra").cast("decimal(10,4)")
)

window_ffill = Window.partitionBy(F.lit(1)).orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_dolar_ffill = (
    df_dolar
    .withColumn("cotacaoCompra", F.last("cotacaoCompra", ignorenulls=True).over(window_ffill))
)

df_dolar_raw = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

ultima_cotacao_linha = df_dolar_raw.orderBy(F.desc("dataHoraCotacao")).first()

if ultima_cotacao_linha is None:
    print("⚠️ ALERTA: Tabela de cotação vazia! Utilizando Dólar fallback de R$ 5.00.")
    cotacao_atual = 5.00
else:
    cotacao_atual = float(ultima_cotacao_linha["cotacaoCompra"])
    data_cotacao_usada = ultima_cotacao_linha["dataHoraCotacao"]
    print(f"💵 Cotação do Dólar utilizada: R$ {cotacao_atual} (Ref: {data_cotacao_usada})")

df_fin_silver = (
    df_fin_silver_parcial
    
    .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)"))
    
    .withColumn("margem_lucro_usd",
        F.when(F.col("receita_usd") > 0, 
               F.round((F.col("lucro_usd") / F.col("receita_usd")) * 100, 2))
         .otherwise(F.lit(None).cast("decimal(18,2)"))
    )
    
    .withColumn("orcamento_brl", (F.col("orcamento_usd") * F.lit(cotacao_atual)).cast("decimal(18,2)"))
    .withColumn("receita_brl", (F.col("receita_usd") * F.lit(cotacao_atual)).cast("decimal(18,2)"))
    .withColumn("lucro_brl", (F.col("lucro_usd") * F.lit(cotacao_atual)).cast("decimal(18,2)"))
)

dq_check_unique("tb_financeiro_filmes", "Unicidade de id_filme", df_fin_silver, ["id_filme"])
dq_check("tb_financeiro_filmes", "Orçamento válido (nulo ou >=0)", df_fin_silver, F.col("orcamento_usd").isNull() | (F.col("orcamento_usd") >= 0))
dq_check("tb_financeiro_filmes", "Receita válida (nula ou >=0)", df_fin_silver, F.col("receita_usd").isNull() | (F.col("receita_usd") >= 0))

tabela_destino_fin = f"{silver_schema}.tb_financeiro_filmes"

(
    df_fin_silver.write
    .format("delta")
    .mode("overwrite") 
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_destino_fin)
)

print(f"✅ Tabela gravada com sucesso em: {tabela_destino_fin}")
display(df_fin_silver.filter(F.col("lucro_usd").isNotNull()).orderBy(F.desc("lucro_usd")).limit(10))


Processando: tb_financeiro_filmes...
💵 Cotação do Dólar utilizada: R$ 5.1111 (Ref: 2026-09-21 13:06:51.445645)
[✅ PASS] tb_financeiro_filmes | Unicidade de id_filme | 0 chaves duplicadas
[✅ PASS] tb_financeiro_filmes | Orçamento válido (nulo ou >=0) | 0/99006 linhas falharam
[✅ PASS] tb_financeiro_filmes | Receita válida (nula ou >=0) | 0/99006 linhas falharam
✅ Tabela gravada com sucesso em: cinedata_analytics.silver.tb_financeiro_filmes


id_filme,orcamento_usd,receita_usd,lucro_usd,margem_lucro_usd,orcamento_brl,receita_brl,lucro_brl
299534,356000000.00,2800000000.00,2444000000.00,87.29,1819551600.00,14311080000.00,12491528400.00
76600,460000000.00,2320250281.00,1860250281.00,80.17,2351106000.00,11859031211.22,9507925211.22
299536,300000000.00,2052415039.00,1752415039.00,85.38,1533330000.00,10490098505.83,8956768505.83
634649,200000000.00,1921847111.00,1721847111.00,89.59,1022220000.00,9822752769.03,8800532769.03
420818,260000000.00,1663075401.00,1403075401.00,84.37,1328886000.00,8500144682.05,7171258682.05
361743,170000000.00,1488732821.00,1318732821.00,88.58,868887000.00,7609062321.41,6740175321.41
346698,145000000.00,1428545028.00,1283545028.00,89.85,741109500.00,7301436492.61,6560326992.61
502356,100000000.00,1355725263.00,1255725263.00,92.62,511110000.00,6929247391.72,6418137391.72
284054,200000000.00,1349926083.00,1149926083.00,85.18,1022220000.00,6899607202.82,5877387202.82
351286,170000000.00,1310466296.00,1140466296.00,87.03,868887000.00,6697924285.49,5829037285.49


## 7. Tratamento: tb_movies_metrics -> tb_metricas_engajamento

- **Deduplicação**: Remove registros com `id` duplicado, mantendo o registro com `ingestion_datetime` mais recente.
- **Limpeza da Popularidade**: A coluna `popularity` apresenta inconsistência de separadores decimais (vírgula vs. ponto). A vírgula é substituída por ponto antes da conversão de tipo.
- **Tipagem Segura (TRY_CAST)**: Devido ao deslocamento de colunas (Column Shift), existem textos fora de contexto nas colunas de notas e contagens. `TRY_CAST` converte apenas valores compatíveis; textos incompatíveis tornam-se `NULL`.
- **Validação de Domínio**: Notas médias (TMDB e IMDb) fora do intervalo 0-10 (incluindo valores multiplicados por erro de escala) são desconsideradas. Contagens de votos e popularidade negativos são invalidados.

In [0]:
print("\nProcessando: tb_metricas_engajamento...")

df_metrics_raw = spark.table(f"{bronze_schema}.tb_movies_metrics")
df_metrics_raw = filter_valid_keys(df_metrics_raw, "id")

df_metrics_dedup = deduplicate_by_ingestion(df_metrics_raw, partition_cols=["id"])

df_metrics_silver = (
    df_metrics_dedup
    .withColumnRenamed("id", "id_filme")
    .withColumn("popularidade", F.expr("TRY_CAST(regexp_replace(popularity, ',', '.') AS DOUBLE)"))
    .withColumn("nota_media_tmdb", F.expr("TRY_CAST(vote_average AS DOUBLE)"))
    .withColumn("qtd_votos_tmdb", F.expr("TRY_CAST(vote_count AS INT)"))
    .withColumn("nota_media_imdb", F.expr("TRY_CAST(averageRating AS DOUBLE)"))
    .withColumn("qtd_votos_imdb", F.expr("TRY_CAST(numVotes AS INT)"))
    .withColumn("nota_media_tmdb", F.when(F.col("nota_media_tmdb").between(0, 10), F.col("nota_media_tmdb")).otherwise(F.lit(None).cast("double")))
    .withColumn("nota_media_imdb", F.when(F.col("nota_media_imdb").between(0, 10), F.col("nota_media_imdb")).otherwise(F.lit(None).cast("double")))
    .withColumn("popularidade", F.when(F.col("popularidade") >= 0, F.col("popularidade")).otherwise(F.lit(None).cast("double")))
    .withColumn("qtd_votos_tmdb", F.when(F.col("qtd_votos_tmdb") >= 0, F.col("qtd_votos_tmdb")).otherwise(F.lit(None).cast("int")))
    .withColumn("qtd_votos_imdb", F.when(F.col("qtd_votos_imdb") >= 0, F.col("qtd_votos_imdb")).otherwise(F.lit(None).cast("int")))
    .select("id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb")
)

dq_check_unique("tb_metricas_engajamento", "Unicidade de id_filme", df_metrics_silver, ["id_filme"])
dq_check("tb_metricas_engajamento", "Notas TMDB 0-10", df_metrics_silver, F.col("nota_media_tmdb").isNull() | F.col("nota_media_tmdb").between(0, 10))
dq_check("tb_metricas_engajamento", "Notas IMDb 0-10", df_metrics_silver, F.col("nota_media_imdb").isNull() | F.col("nota_media_imdb").between(0, 10))
dq_check("tb_metricas_engajamento", "Popularidade nao-negativa", df_metrics_silver, F.col("popularidade").isNull() | (F.col("popularidade") >= 0))

tabela_destino_metrics = f"{silver_schema}.tb_metricas_engajamento"
(
    df_metrics_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_destino_metrics)
)
print(f"✅ Tabela gravada com sucesso em: {tabela_destino_metrics}")
display(df_metrics_silver.limit(10))


Processando: tb_metricas_engajamento...
[✅ PASS] tb_metricas_engajamento | Unicidade de id_filme | 0 chaves duplicadas
[✅ PASS] tb_metricas_engajamento | Notas TMDB 0-10 | 0/95261 linhas falharam
[✅ PASS] tb_metricas_engajamento | Notas IMDb 0-10 | 0/95261 linhas falharam
[✅ PASS] tb_metricas_engajamento | Popularidade nao-negativa | 0/95261 linhas falharam
✅ Tabela gravada com sucesso em: cinedata_analytics.silver.tb_metricas_engajamento


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1000073,13.212,6.8,15,null,236
1000081,102.802,5.142,134,5.1,1841
1000092,4.797,null,null,null,130
1000094,2.645,6.9,28,6.5,null
1000096,0.71,10.0,1,6.3,695
1000108,3.231,0.0,0,6.5,157
1000156,1.193,8.0,null,6.7,null
1000172,1.662,null,2,6.7,398
1000176,1.075,7.1,5,null,277
1000194,6.191,6.554,57,7.0,1652


## 8. Tratamento: tb_movies_reviews -> tb_avaliacoes_usuarios

- **Renomeação de Colunas**: Mapeamento dos nomes originais para termos de negócio em português.
- **Remoção de Duplicatas Integrais**: Registros onde a combinação de filme, usuário, nota e comentário seja idêntica são removidos.
- **Validação de Nota**: A nota do usuário deve respeitar a escala permitida (0 a 10). Valores fora dessa faixa são convertidos para `NULL`.
- **Preenchimento de Comentários Vazios**: Comentários não preenchidos ou compostos apenas por espaços em branco são preenchidos com o texto padronizado "Sem comentário".

In [0]:
print("\nProcessando: tb_avaliacoes_usuarios...")

df_reviews_raw = spark.table(f"{bronze_schema}.tb_movies_reviews")
df_reviews_raw = filter_valid_keys(df_reviews_raw, "id")

df_reviews = (
    df_reviews_raw
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario_raw")
    .withColumnRenamed("comentario", "comentario_usuario")
    .withColumn("nota_usuario", F.expr("TRY_CAST(nota_usuario_raw AS DOUBLE)"))
    .withColumn("nota_usuario", F.when(F.col("nota_usuario").between(0, 10), F.col("nota_usuario")).otherwise(F.lit(None).cast("double")))
    .drop("nota_usuario_raw", "ingestion_datetime")
    .withColumn("comentario_usuario", F.when(F.col("comentario_usuario").isNull() | (F.regexp_replace(F.col("comentario_usuario"), r"\\s", "") == ""), F.lit("Sem comentário")).otherwise(F.col("comentario_usuario")))
)

df_reviews_silver = df_reviews.dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])

dq_check_unique("tb_avaliacoes_usuarios", "Unicidade de avaliacao", df_reviews_silver, ["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
dq_check("tb_avaliacoes_usuarios", "Nota no intervalo 0-10", df_reviews_silver, F.col("nota_usuario").isNull() | F.col("nota_usuario").between(0, 10))

tabela_destino_reviews = f"{silver_schema}.tb_avaliacoes_usuarios"
(
    df_reviews_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_destino_reviews)
)
print(f"✅ Tabela gravada com sucesso em: {tabela_destino_reviews}")
display(df_reviews_silver.limit(10))


Processando: tb_avaliacoes_usuarios...
[✅ PASS] tb_avaliacoes_usuarios | Unicidade de avaliacao | 0 chaves duplicadas
[✅ PASS] tb_avaliacoes_usuarios | Nota no intervalo 0-10 | 0/32412 linhas falharam
✅ Tabela gravada com sucesso em: cinedata_analytics.silver.tb_avaliacoes_usuarios


id_filme,nome_usuario,comentario_usuario,nota_usuario
637007,Lucas Reis 602,Sem comentário,3.9
1100094,Gabriel Carvalho 581,"Aceitável, mas esperava mais.",6.2
628575,Alexandre Barbosa 220,Péssimo em todos os sentidos.,0.3
573249,Rodrigo Oliveira 273,Péssimo em todos os sentidos.,0.5
592539,Pedro Costa 181,"Não gostei, história confusa.",4.8
464493,Adriana Dias 257,Péssimo em todos os sentidos.,0.4
1199748,Eduardo Dias 177,"Poderia ser melhor, mas não é ruim.",6.0
640543,Cristina Monteiro 310,Sem comentário,6.4
599134,Larissa Lopes 330,Péssimo em todos os sentidos.,2.6
424011,Vinícius Ferreira 405,Não recomendo de jeito nenhum.,0.5


## 9. Tratamento: tb_credits_and_tags -> tb_generos

### Pipeline da coluna `genres` -> silver.tb_generos

- **Normalização de Separadores**: vírgula, ponto-e-vírgula e pipe são normalizados para vírgula antes do `split`.
- **Split + Explode**: desmembra os valores para que cada registro represente um único gênero por filme.
- **Remoção de Resíduos (Column Shift)**: paths de imagem, valores numéricos, sentinelas e textos descritivos deslocados são filtrados.

### Justificativas da Análise Exploratória

As decisões de filtragem foram baseadas em **investigação empírica dos dados reais** da camada Bronze.

**1. Separadores `/` e `\` não são delimitadores**

Investigação encontrou 1.139 linhas com `/` e 525 com `\` — **nenhuma é separador de gênero**. `/` aparece em paths de imagem (`/oajNi4Su...jpg`) e `\` em aspas escapadas (`\"`) de sinopses deslocadas. Conclusão: não há necessidade de normalizá-los como separadores.

**2. Os regex não capturam tudo — a whitelist é a garantia definitiva**

Após aplicar todos os regex de limpeza, **1.663 resíduos** ainda escapam (paths `.JPG` maiúsculos, números com aspas coladas, países, nomes de pessoas). Os regex são camadas de otimização que reduzem volume, mas a whitelist barra 100% dos resíduos restantes.

**3. A whitelist é necessária — heurísticas não substituem conhecimento de domínio**

Após aplicar **todas as heurísticas possíveis** (paths, números, sentinelas, alfabético-only, máximo 2 palavras, limite de tamanho), sobram **480 valores distintos**: 19 gêneros válidos e **461 resíduos indistinguíveis**. `"Action"` e `"France"` têm o mesmo padrão estrutural (1 palavra alfabética). Uma blocklist seria infinita (todos os países, idiomas, nomes...); a whitelist é finita (19 gêneros oficiais do TMDB) e estável, assim como não sabemos qual a API oficial para poder consultar os valores de gênero oficiais.



In [0]:
print("\nProcessando: tb_generos...")

df_credits_raw = spark.table(f"{bronze_schema}.tb_credits_and_tags")
df_credits_raw = filter_valid_keys(df_credits_raw, "id")

df_credits_dedup = deduplicate_by_ingestion(df_credits_raw, partition_cols=["id"])

df_generos_silver = (
    df_credits_dedup
    .select(F.col("id").alias("id_filme"), F.col("genres"))
    .withColumn("genres_norm", F.regexp_replace(F.regexp_replace(F.col("genres"), r"\|", ","), r";", ","))
    .withColumn("genero", F.explode(F.split("genres_norm", ",")))
    .withColumn("nome_genero", F.trim(F.col("genero")))
    .filter(F.col("nome_genero") != "")
    .filter(~F.col("nome_genero").rlike(r"(?i)^/.*\.(jpe?g|png|gif|bmp|webp|svg|tiff?)$"))
    .filter(~F.col("nome_genero").rlike(r"^\d+$"))
    .filter(~F.col("nome_genero").rlike(r"^\d+\.\d+$"))
    .filter(~F.col("nome_genero").isin(["[]", "N/A", "Nenhum", "null", "none"]))
    .filter(F.length(F.col("nome_genero")) <= 50)
    .filter(F.lower(F.col("nome_genero")).isin([
        "action", "adventure", "animation", "comedy", "crime", "documentary",
        "drama", "family", "fantasy", "history", "horror", "music",
        "mystery", "romance", "science fiction", "tv movie", "thriller",
        "war", "western"
    ]))
    .withColumn("nome_genero",
        F.when(F.lower(F.col("nome_genero")) == "science fiction", F.lit("Science Fiction"))
         .when(F.lower(F.col("nome_genero")) == "tv movie", F.lit("TV Movie"))
         .otherwise(F.initcap(F.col("nome_genero")))
    )
    .select("id_filme", "nome_genero")
    .dropDuplicates(["id_filme", "nome_genero"])
)

dq_check_unique("tb_generos", "Unicidade id_filme + nome_genero", df_generos_silver, ["id_filme", "nome_genero"])

tabela_destino_generos = f"{silver_schema}.tb_generos"
(
    df_generos_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_destino_generos)
)
print(f"✅ Tabela gravada com sucesso em: {tabela_destino_generos}")
display(df_generos_silver.limit(20))


Processando: tb_generos...
[✅ PASS] tb_generos | Unicidade id_filme + nome_genero | 0 chaves duplicadas
✅ Tabela gravada com sucesso em: cinedata_analytics.silver.tb_generos


id_filme,nome_genero
1000005,Comedy
1000007,Comedy
1000014,Comedy
1000075,Action
1000075,Drama
1000075,Adventure
1000079,Comedy
1000079,Drama
1000092,Animation
1000094,Comedy


## 10. Tratamentot b_credits_and_tags -> tb_pessoas_empresas

- **Dimensão Unificada**: Consolida quatro tipos de entidade em uma única tabela, categorizada por `tipo_entidade` (Ator, Diretor, Roteirista, Produtora).
- **Limpeza de Sentinelas**: Valores como `N/A`, `[]`, `Nenhum` e `null` são removidos antes do split.
- **Padronização de Capitalização**: Aplicação de `initcap` para padronizar nomes próprios e nomes de produtoras.
- **Deduplicação**: Remoção de registros duplicados por (`id_filme`, `nome_entidade`, `tipo_entidade`).

In [0]:
print("\nProcessando: tb_pessoas_empresas...")

df_atores = explode_entities(df_credits_dedup, "cast", "Ator")
df_diretores = explode_entities(df_credits_dedup, "directors", "Diretor")
df_roteiristas = explode_entities(df_credits_dedup, "writers", "Roteirista")
df_produtoras = explode_entities(df_credits_dedup, "production_companies", "Produtora")

df_pessoas_empresas_silver = (
    df_atores
    .union(df_diretores)
    .union(df_roteiristas)
    .union(df_produtoras)
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

dq_check_unique("tb_pessoas_empresas", "Unicidade id_filme + nome_entidade + tipo_entidade",
    df_pessoas_empresas_silver, ["id_filme", "nome_entidade", "tipo_entidade"])

tabela_destino_pessoas = f"{silver_schema}.tb_pessoas_empresas"
(
    df_pessoas_empresas_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_destino_pessoas)
)
print(f"✅ Tabela gravada com sucesso em: {tabela_destino_pessoas}")
display(df_pessoas_empresas_silver.limit(20))


Processando: tb_pessoas_empresas...
[✅ PASS] tb_pessoas_empresas | Unicidade id_filme + nome_entidade + tipo_entidade | 0 chaves duplicadas
✅ Tabela gravada com sucesso em: cinedata_analytics.silver.tb_pessoas_empresas


id_filme,nome_entidade,tipo_entidade
1000079,Camille Rutherford,Ator
1000088,Laurence Côte,Ator
1000938,Dharma Mangia Woods,Ator
1001826,Jason Haines,Ator
1003431,Isa Aouifia,Ator
1004495,Jocelyn Deboer,Ator
1005342,John Mcafee,Ator
1006242,Yoël Rozenkier,Ator
1006267,Cindy Busby,Ator
1007097,Charles Murphy Iwuchukwu,Ator


## 10. Persistência do Log de Data Quality (Observabilidade)

O micro-framework de Data Quality coleta resultados ao longo de toda a execução via `dq_check` e `dq_check_unique`. Aqui, persistimos o histórico em uma tabela física (`silver.dq_log`) utilizando `.mode("append")`, fornecendo uma trilha de auditoria contínua que atesta não apenas que o dado foi modelado, mas que sua estrutura lógica não sofreu corrupção ao longo do tempo.

In [0]:
print("\nPersistindo Log de Data Quality...")

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, TimestampType

dq_schema = StructType([
    StructField("table_name", StringType(), True),
    StructField("check_name", StringType(), True),
    StructField("total_rows", IntegerType(), True),
    StructField("failed_rows", IntegerType(), True),
    StructField("passed", BooleanType(), True),
    StructField("checked_at", TimestampType(), True)
])

if dq_results:
    df_dq_log = spark.createDataFrame(dq_results, schema=dq_schema)
    tabela_dq_log = f"{silver_schema}.dq_log"
    (
        df_dq_log.write
        .format("delta")
        .mode("append")
        .saveAsTable(tabela_dq_log)
    )
    print(f"📋 Log de Data Quality persistido em: {tabela_dq_log}")
    display(df_dq_log)
else:
    print("⚠️ Nenhum resultado de DQ para persistir.")


Persistindo Log de Data Quality...
📋 Log de Data Quality persistido em: cinedata_analytics.silver.dq_log


table_name,check_name,total_rows,failed_rows,passed,checked_at
tb_cotacao_dolar,Unicidade de data_referencia,48,0,true,2026-09-21T20:51:26.422Z
tb_cotacao_dolar,Cotação válida (nula ou > 0),48,0,true,2026-09-21T20:51:28.349Z
tb_info_filmes,Unicidade de id_filme,93220,0,true,2026-09-21T20:51:37.156Z
tb_info_filmes,Duração válida (nula ou >=0),93220,0,true,2026-09-21T20:51:38.797Z
tb_info_filmes,Ano de lançamento válido (nulo ou >=1900),93220,0,true,2026-09-21T20:51:40.666Z
tb_financeiro_filmes,Unicidade de id_filme,99006,0,true,2026-09-21T20:51:47.832Z
tb_financeiro_filmes,Orçamento válido (nulo ou >=0),99006,0,true,2026-09-21T20:51:49.774Z
tb_financeiro_filmes,Receita válida (nula ou >=0),99006,0,true,2026-09-21T20:51:51.747Z
tb_metricas_engajamento,Unicidade de id_filme,95261,0,true,2026-09-21T20:52:00.452Z
tb_metricas_engajamento,Notas TMDB 0-10,95261,0,true,2026-09-21T20:52:02.249Z
